# Trainings- und Validierungsverläufe der Finetune-Rejectoren

Gemeinsamer Full-Width-Zweierplot mit $r_1$ links und $r_2$ rechts.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, MultipleLocator, PercentFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
})

PROJECT_ROOT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training")
PLOTS_DIR = PROJECT_ROOT / "plot"
OUT_DIR = PLOTS_DIR / "figures_rejector_finetune"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_ROOTS = {
    "r1": PROJECT_ROOT / "tensorboard_runs/tensorboard_runs_rejector_r1_FINETUNE/embedding/conv_mlp",
    "r2": PROJECT_ROOT / "tensorboard_runs/tensorboard_runs_rejector_r2_FINETUNE/embedding/conv_mlp",
}

PDF_PATH = OUT_DIR / "rejector_finetune_train_val_accuracy.pdf"

FIGSIZE_FULL = (6.8, 2.3)
MAX_EPOCH = None
YLIM = (0.50, 1.00)
YTICK_STEP = 0.10


In [ ]:
def read_scalar_events(scalar_dir: Path) -> list[dict]:
    if not scalar_dir.exists():
        return []

    event_files = sorted(scalar_dir.glob("events.out.tfevents.*"))
    if not event_files:
        return []

    accumulator = EventAccumulator(str(scalar_dir), size_guidance={"scalars": 0})
    accumulator.Reload()

    tags = accumulator.Tags().get("scalars", [])
    if not tags:
        return []

    preferred_tag = scalar_dir.name
    tag = preferred_tag if preferred_tag in tags else tags[0]

    rows = []
    for event in accumulator.Scalars(tag):
        rows.append({
            "step": int(event.step),
            "value": float(event.value),
            "wall_time": float(event.wall_time),
            "tag": tag,
        })
    return rows


def load_stage(stage: str, root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    scalar_rows = []
    run_rows = []

    if not root.exists():
        warnings.warn(f"Run-Root fehlt: {root}")
        return pd.DataFrame(), pd.DataFrame()

    run_dirs = sorted(path for path in root.iterdir() if path.is_dir() and path.name.startswith("run_embedding"))

    for run_dir in run_dirs:
        split_counts = {}
        for split, scalar_name in [("train", "Accuracy_train"), ("val", "Accuracy_val")]:
            events = read_scalar_events(run_dir / scalar_name)
            split_counts[split] = len(events)
            for event in events:
                scalar_rows.append({
                    "stage": stage,
                    "run": run_dir.name,
                    "run_dir": str(run_dir),
                    "split": split,
                    **event,
                })

        run_rows.append({
            "stage": stage,
            "run": run_dir.name,
            "run_dir": str(run_dir),
            "n_train": split_counts.get("train", 0),
            "n_val": split_counts.get("val", 0),
        })

    scalars = pd.DataFrame(scalar_rows)
    runs = pd.DataFrame(run_rows)

    if scalars.empty:
        return scalars, runs

    # TensorBoard can contain duplicated steps after resumed runs. Keep the latest value per step.
    scalars = (
        scalars.sort_values("wall_time")
        .drop_duplicates(["stage", "run", "split", "step"], keep="last")
        .sort_values(["stage", "run", "split", "step"])
        .reset_index(drop=True)
    )

    return scalars, runs


loaded = [load_stage(stage, root) for stage, root in RUN_ROOTS.items()]
scalars = pd.concat([item[0] for item in loaded if not item[0].empty], ignore_index=True)
runs = pd.concat([item[1] for item in loaded if not item[1].empty], ignore_index=True)

summary = (
    scalars.loc[scalars["split"] == "val"]
    .groupby(["stage", "run"], as_index=False)
    .agg(best_val_accuracy=("value", "max"), max_epoch=("step", "max"))
)

print(f"Geladene Scalar-Punkte: {len(scalars)}")
display(runs)
display(summary)


In [ ]:
RUN_ORDER = {
    stage: runs.loc[runs["stage"] == stage, "run"].tolist()
    for stage in RUN_ROOTS
}

colors = plt.colormaps["tab20"].resampled(
    max(max((len(run_names) for run_names in RUN_ORDER.values()), default=1), 1)
)

RUN_COLORS = {
    stage: {run: colors(index) for index, run in enumerate(run_names)}
    for stage, run_names in RUN_ORDER.items()
}

GLOBAL_MAX_STEP = scalars["step"].max() if not scalars.empty else np.nan


def style_axis(ax, ylabel=None, ylim=None, ytick_step=None, percent=False):
    ax.set_xlabel("Epoche", labelpad=2)

    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=2)

    if ylim is not None:
        ax.set_ylim(*ylim)

    if ytick_step is not None:
        ax.yaxis.set_major_locator(MultipleLocator(ytick_step))

    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

    ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.grid(axis="both", color="0.90", linewidth=0.7, linestyle="-")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def set_epoch_xlim(axes):
    right = MAX_EPOCH if MAX_EPOCH is not None else GLOBAL_MAX_STEP
    if pd.notna(right):
        for ax in np.ravel(axes):
            ax.set_xlim(0, right + max(1, right * 0.02))


def save_figure(fig, path: Path):
    fig.savefig(path)
    print(path)


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=FIGSIZE_FULL,
    sharex=True,
    sharey=True,
)

stage_specs = [
    ("r1", r"$r_1$"),
    ("r2", r"$r_2$"),
]

for ax, (stage, stage_label) in zip(axes, stage_specs):
    stage_frame = scalars.loc[scalars["stage"] == stage]
    train_frame = stage_frame.loc[stage_frame["split"] == "train"]
    val_frame = stage_frame.loc[stage_frame["split"] == "val"]

    for run in RUN_ORDER[stage]:
        color = RUN_COLORS[stage][run]

        run_train = train_frame.loc[train_frame["run"] == run].sort_values("step")
        run_val = val_frame.loc[val_frame["run"] == run].sort_values("step")

        if not run_train.empty:
            ax.plot(
                run_train["step"],
                run_train["value"],
                color=color,
                linestyle="--",
                linewidth=0.8,
                alpha=0.65,
            )

        if not run_val.empty:
            ax.plot(
                run_val["step"],
                run_val["value"],
                color=color,
                linestyle="-",
                linewidth=0.9,
                alpha=0.85,
            )

    ax.set_title(stage_label, pad=3)

    style_axis(
        ax,
        ylabel="Genauigkeit" if ax is axes[0] else None,
        ylim=YLIM,
        ytick_step=YTICK_STEP,
        percent=True,
    )

set_epoch_xlim(axes)

style_handles = [
    Line2D([0], [0], color="0.25", linestyle="--", linewidth=0.9, label="Training"),
    Line2D([0], [0], color="0.25", linestyle="-", linewidth=0.9, label="Validierung"),
]

fig.legend(
    handles=style_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=2,
    frameon=False,
    fontsize=7,
    handlelength=1.5,
    handletextpad=0.4,
    columnspacing=1.2,
)

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.91,
    bottom=0.26,
    wspace=0.18,
)

save_figure(fig, PDF_PATH)
plt.show()
